# Epoch 1-3 Vocal/Noise Long-Form Sweep

This notebook runs the same 5 random Downloads songs through epochs 1, 2, and 3 of the latest vocal/crackle diffusion retool run.

The settings panel is focused around the `low_noise_content` and `vocal_safe` region, with 20 variants tuned to reduce crackling and static while still preserving some style movement.

Design goal:
- same songs across all epochs
- same offsets across all epochs
- same 20 settings across all epochs
- compare checkpoint quality against the long-form dial regime directly

In [ ]:
from pathlib import Path
import importlib
import json
import sys

import pandas as pd

def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'dggr').exists() and (path / 'lab 4').exists() and (path / 'lab 3.1').exists():
            return path
    raise RuntimeError('Could not resolve repo root from current working directory.')

REPO = find_repo_root(Path.cwd().resolve())
SCRIPTS = REPO / 'lab 3.1' / 'scripts'
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

import diffusion_longform_settings_sweep as sweep
importlib.reload(sweep)
REPO

In [ ]:
def latest_vocal_crackle_run(root: Path) -> Path:
    runs = [p for p in root.iterdir() if p.is_dir() and (p / 'checkpoints').exists()]
    if not runs:
        raise RuntimeError('No diffusion_vocal_crackle_retool runs found.')
    runs.sort(key=lambda p: (p / 'checkpoints').stat().st_mtime if (p / 'checkpoints').exists() else p.stat().st_mtime, reverse=True)
    return runs[0]

RUN_DIR = latest_vocal_crackle_run(REPO / 'lab 3.1' / 'outputs' / 'diffusion_vocal_crackle_retool')
CHECKPOINTS = sweep.resolve_epoch_panel(RUN_DIR, ('epoch_001.pt', 'epoch_002.pt', 'epoch_003.pt'))
if not CHECKPOINTS:
    raise RuntimeError('No epoch_001/002/003 checkpoints found in the latest vocal/crackle run.')

BASE_TAG = Path(RUN_DIR).name + '_vocal_noise_panel'
OUTPUT_ROOT = REPO / 'lab 3.1' / 'outputs' / 'diffusion_longform_vocal_noise_panel'

print('Run dir:')
print(RUN_DIR)
print('\nEpoch checkpoints:')
for ckpt in CHECKPOINTS:
    print(' -', ckpt)
print('\nOutput root:')
print(OUTPUT_ROOT)

In [ ]:
base_cfg = sweep.DiffusionSettingsSweepConfig(
    downloads_dir=Path.home() / 'Downloads',
    output_root=OUTPUT_ROOT,
    run_dir=RUN_DIR,
    cache_dir=REPO / 'saves2' / 'lab3_diffusion' / 'run_d001' / 'cache',
    n_songs=5,
    targets_per_song=1,
    source_seconds=45.0,
    seed=328,
)

RUN_ALL = True

settings_panel = sweep.vocal_low_noise_settings_panel()
display(pd.DataFrame(settings_panel))
print('Total settings:', len(settings_panel))

In [ ]:
plan_cfg = sweep.dlc.DiffusionLongformCompareConfig(
    downloads_dir=base_cfg.downloads_dir,
    output_root=base_cfg.output_root,
    run_dir=base_cfg.run_dir,
    cache_dir=base_cfg.cache_dir,
    lab1_checkpoint=base_cfg.lab1_checkpoint,
    n_songs=base_cfg.n_songs,
    targets_per_song=base_cfg.targets_per_song,
    source_seconds=base_cfg.source_seconds,
    chunk_seconds=base_cfg.chunk_seconds,
    overlap_seconds=base_cfg.overlap_seconds,
    n_frames=base_cfg.n_frames,
    ddim_steps=base_cfg.ddim_steps,
    assemble_domain=base_cfg.assemble_domain,
    device=base_cfg.device,
    seed=base_cfg.seed,
    snapshot_latest_checkpoint=False,
)
jobs = sweep.dlc.plan_longform_jobs(plan_cfg)
display(pd.DataFrame(jobs))
print('Songs:', base_cfg.n_songs)
print('Jobs per epoch:', len(jobs))
print('Total runs across epochs/settings:', len(jobs) * len(settings_panel) * len(CHECKPOINTS))

In [ ]:
summaries = []
if RUN_ALL:
    for ckpt in CHECKPOINTS:
        cfg = sweep.DiffusionSettingsSweepConfig(
            tag=f'{BASE_TAG}_{ckpt.stem}',
            downloads_dir=base_cfg.downloads_dir,
            output_root=base_cfg.output_root,
            run_dir=base_cfg.run_dir,
            cache_dir=base_cfg.cache_dir,
            checkpoint_path=ckpt,
            lab1_checkpoint=base_cfg.lab1_checkpoint,
            n_songs=base_cfg.n_songs,
            targets_per_song=base_cfg.targets_per_song,
            source_seconds=base_cfg.source_seconds,
            chunk_seconds=base_cfg.chunk_seconds,
            overlap_seconds=base_cfg.overlap_seconds,
            n_frames=base_cfg.n_frames,
            ddim_steps=base_cfg.ddim_steps,
            assemble_domain=base_cfg.assemble_domain,
            device=base_cfg.device,
            seed=base_cfg.seed,
        )
        summary = sweep.run_settings_sweep(cfg, settings_panel)
        summaries.append(summary)
    print(json.dumps(summaries, indent=2, default=str))
else:
    print('Set RUN_ALL = True to launch the epoch 1-3 vocal/noise sweep.')

In [ ]:
rows = []
for ckpt in CHECKPOINTS:
    tag = f'{BASE_TAG}_{ckpt.stem}'
    manifest_path = OUTPUT_ROOT / tag / 'manifest.csv'
    if manifest_path.exists():
        df = pd.read_csv(manifest_path)
        df.insert(0, 'epoch_label', ckpt.stem)
        rows.append(df)

if rows:
    combined = pd.concat(rows, ignore_index=True)
    display(combined.head(30))
    print('Combined rows:', len(combined))
else:
    print('No manifests yet.')